In [ ]:
import requests
import json
from datetime import date

import pandas as pd

In [ ]:
baseURL = "https://catalogue.aodn.org.au/geonetwork/srv/eng/q?"
params = {
    "title": "NESP",
    "type": "dataset",
    "_content_type": "json",
    "fast": "index",
    "from": 1,
    "resultType": "details"
}

today = date.today().strftime("%Y%m%d")

##https://catalogue.aodn.org.au/geonetwork/srv/eng/q?title=NESP&type=dataset&_content_type=json&fast=index&from=1&resultType=details

In [ ]:
## make a query
response = requests.get(baseURL, params=params)
response.raise_for_status()

## parse the response
data = response.json()

## convert to pandas df
df = pd.DataFrame(data["metadata"])

## extract id and uuid from geonet:info dict
df["uuid"] = df["geonet:info"].apply(lambda x: x["uuid"])
df["id"] = df["geonet:info"].apply(lambda x: x["id"])

## get the total number of records
nRecords = int(data['summary']['@count'])

## iterate over the rest of the pages
page = 2
for i in range(1, int(nRecords/100) + 1):
    params["from"] = i*100 + 1
    response = requests.get(baseURL, params=params)
    response.raise_for_status()
    data = response.json()
    dfTemp = pd.DataFrame(data["metadata"])
    dfTemp["uuid"] = dfTemp["geonet:info"].apply(lambda x: x["uuid"])
    dfTemp["id"] = dfTemp["geonet:info"].apply(lambda x: x["id"])
    df = pd.concat([df, dfTemp], ignore_index=True)
    print(page)
    page += 1

In [ ]:
## select columns
df = df[['id', 'uuid', 'type', 'title', 'abstract', 'keyword', 'geoPolygon', 'geoBox', 'tempExtentBegin', 'tempExtentEnd', 'creationDate']]

In [5]:
len(df)

193

In [ ]:
## save to csv
df.to_csv(f"NESP_datasets_{today}.csv", index=False)

In [294]:
df = df.loc[df['title'].str.contains("MaC", na=False)]
print(len(df))

86


In [ ]:
## extract coordinates for geoBox
def geobox_to_wkt(geobox):
    if type(geobox) is str: 
        minlon, minlat, maxlon, maxlat = geobox.split("|")
        wkt = f"POLYGON(({minlon} {minlat}, {maxlon} {minlat}, {maxlon} {maxlat}, {minlon} {maxlat}, {minlon} {minlat}))"
        return wkt
    if type(geobox) is list:
        temp = "|".join(geobox)
        if "POINT" in temp:
            wkt = "MULTIPOINT("
            for i in range(len(geobox)):
                lon, lat = str(geobox[i]).split("|")
                wkt += f"({lon} {lat}),"
            wkt = wkt[:-1] + ")"
            return wkt
        else:
            wkt = "MULTIPOLYGON("
            for i in range(len(geobox)):
                print(geobox[i])
                minlon, minlat, maxlon, maxlat = str(geobox[i]).split("|")
                wkt += f"(({minlon} {minlat}, {maxlon} {minlat}, {maxlon} {maxlat}, {minlon} {maxlat}, {minlon} {minlat})),"
            wkt = wkt[:-1] + ")"    
            return wkt
    else:
        return None

In [282]:
## extract coordinates for geoBox
def geobox_to_wkt(geobox):
    geobox = geobox[0]
    if type(geobox) is str: 
        minlon, minlat, maxlon, maxlat = geobox.split("|")
        wkt = f"POLYGON(({minlon} {minlat}, {maxlon} {minlat}, {maxlon} {maxlat}, {minlon} {maxlat}, {minlon} {minlat}))"
        return wkt
    if type(geobox) is list:
        temp = "|".joint(geobox)
        if "POINT" in temp:
            print("POINT")
            wkt = "MULTIPOINT("
            for i in range(len(geobox)):
                lon, lat = str(geobox[i]).split("|")
                wkt += f"({lon} {lat}),"
            wkt = wkt[:-1] + ")"
            return wkt
        else:
            print ("POLYGON")
            wkt = "MULTIPOLYGON("
            for i in range(len(geobox)):
                print(geobox[i])
                minlon, minlat, maxlon, maxlat = str(geobox[i]).split("|")
                wkt += f"(({minlon} {minlat}, {maxlon} {minlat}, {maxlon} {maxlat}, {minlon} {maxlat}, {minlon} {minlat})),"
            wkt = wkt[:-1] + ")"    
            return wkt
    else:
        return None




In [ ]:

df["wkt"] = None
for i in range(len(df)):
    print(i)
    print(df['geoPolygon'][i])
    if type(df['geoPolygon'][i]) is float:
        if type(df['geoBox'][i]) is float:
            df.loc[i, "wkt"] = None
        else:
            print("BBOX", i, df['geoBox'].loc[i])
            bbox = df['geoBox'][i]
            if type(bbox) is list:
                kk = [bbox]
            wkt = geobox_to_wkt(kk)
            #print(wkt)
            df.loc[i, "wkt"] = wkt
    else:
        if df['geoPolygon'][i] == "MULTIPOLYGON EMPTY":
            df.loc[i, "wkt"] = None
        else:
            print("geoPoly", i, df['geoPolygon'][i])
            df.loc[i, "wkt"] = df['geoPolygon'][i]
        





In [ ]:
df['keyword'].to_csv(f"keywords_AODN_{today}.csv")

In [ ]:
## calculate the temporal extent in decimal years using tempExtentBegin and tempExtentEnd
df["temporalExtent"] = None
for i in range(len(df)):
    if df["tempExtentBegin"][i] is not None and df["tempExtentEnd"][i] is not None:
        tempExtentBegin = pd.to_datetime(df["tempExtentBegin"][i])
        tempExtentEnd = pd.to_datetime(df["tempExtentEnd"][i])
        tempExtent = tempExtentEnd - tempExtentBegin
        df.loc[i, "temporalExtent"] = tempExtent.days/365.25
    else:
        df.loc[i, "temporalExtent"] = None

In [ ]:
## check for wrong temporal extents
df_wrongExtent = df.loc[df['temporalExtent'] < 0]

## save to csv if any found
if len(df_wrongExtent) > 0:
    df_wrongExtent.to_csv(f"AODN_wrongTemporalExtent_{today}.csv")

In [256]:

kk = (df['geoPolygon'][i])
print(kk)
temp = "|".join(kk)
"POINT" in temp

['POINT (142.1687949896024 -9.971317449558569)', 'POINT (142.34357292493738 -9.808821980495367)']


True

In [ ]:
## save the table as csv
df.to_csv(f"NESP_MaC_datasets_{today}.csv", index=False)

In [16]:
## save the table as csv
## only the record with a valid geoPolygon
## and with NESP MaC in the title
df = df[df["geoPolygon"].notnull()]
df = df[df["title"].str.contains("NESP MaC")]
df.to_csv("NESP_MaC_datasets.csv", index=False)




In [ ]:
## make a map of all records using folium and geopandas

import folium
import geopandas as gpd
from shapely import wkt as shapely_wkt

## filter to records with valid geometry
df_map = df[df["wkt"].notnull()]

## create a map
m = folium.Map(location=[-27, 133], zoom_start=4)

## iterate over the records
for idx, row in df_map.iterrows():
    ## convert the WKT string to a shapely object
    poly = shapely_wkt.loads(row["wkt"])
    ## convert to geopandas dataframe
    gdf = gpd.GeoDataFrame(geometry=[poly])
    ## add to the map
    folium.GeoJson(gdf).add_to(m)

m.save("map.html")

In [ ]:
## make a map of all records using folium and geopandas

import folium
import geopandas as gpd
from shapely.geometry import shape

## create a map
m = folium.Map(location=[-27, 133], zoom_start=4)

## iterate over the records
## geopolygon is a string that defines a wkt object
for idx, row in df.iterrows():
    if row["geoPolygon"]:
        ## convert the string to a shapely object
        poly = shape(json.loads(row["geoPolygon"]))
        ## convert to geopandas dataframe
        gdf = gpd.GeoDataFrame(geometry=[poly])
        ## add to the map
        folium.GeoJson(gdf).add_to(m)

m.save("map.html")
